# DataGuard: Auto-Fix Functions

## Notebook Purpose

This notebook is the fifth notebook in the DataGuard project.

The goal of this notebook is to create reusable cleaning functions that fix simple, well-understood data quality issues in the NYC 311 dataset.

This notebook focuses on:
- creating a cleaned copy of the raw dataset
- standardizing text columns
- parsing date columns
- cleaning ZIP code formatting
- removing duplicate records safely
- creating resolution-time fields
- documenting each fix in an auto-fix report

## Important Rule

The raw dataset will not be modified directly.

All cleaning will be applied to a copied DataFrame called `df_clean`.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

print("Imports complete.")

Imports complete.


## Load Raw Dataset

In this step, I load the original NYC 311 raw dataset.

The raw DataFrame will be named `df`.
Cleaning will not be applied directly to this DataFrame.

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "nyc_311_raw_100k.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)

df.head()

Dataset loaded successfully.
Rows and columns: (100000, 44)


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,69486831,2026-06-26T02:06:32.000,NaN,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,NaN,Club/Bar/Restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,In Progress,NaN,NaN,NaN,0 Unspecified,NaN,Unspecified,NaN,Unspecified,991542.0,189307.0,ONLINE,Unspecified,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.686278,-73.973707,POINT (-73.9737071634 40.686277751362)
1,69492768,2026-06-26T02:06:05.000,NaN,TLC,Taxi and Limousine Commission,For Hire Vehicle Complaint,Driver Complaint - Non Passenger,Unsafe Driving,Street,11211.0,180 UNION AVENUE,UNION AVENUE,BROADWAY,JOHNSON AVENUE,BROADWAY,JOHNSON AVENUE,ADDRESS,BROOKLYN,UNION AVENUE,NaN,In Progress,NaN,NaN,NaN,01 BROOKLYN,34.0,Precinct 90,3.030580e+09,BROOKLYN,998041.0,196408.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,"180 UNION AVENUE, BROOKLYN, NY, 11211",NaN,NaN,NaN,NaN,40.705761,-73.950259,POINT (-73.950259119641 40.705760603484)
2,69482343,2026-06-26T02:05:24.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10013.0,118 MULBERRY STREET,MULBERRY STREET,CANAL STREET,HESTER STREET,CANAL STREET,HESTER STREET,ADDRESS,NEW YORK,MULBERRY STREET,NaN,In Progress,NaN,NaN,NaN,02 MANHATTAN,1.0,Precinct 5,1.002050e+09,MANHATTAN,984753.0,200765.0,ONLINE,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.717730,-73.998185,POINT (-73.998185472064 40.717730264942)
3,69488306,2026-06-26T02:03:49.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11249.0,96 WYTHE AVENUE,WYTHE AVENUE,NORTH 11 STREET,NORTH 10 STREET,NORTH 11 STREET,NORTH 10 STREET,ADDRESS,BROOKLYN,WYTHE AVENUE,NaN,In Progress,NaN,NaN,NaN,01 BROOKLYN,33.0,Precinct 94,3.022950e+09,BROOKLYN,995800.0,202137.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721489,-73.958332,POINT (-73.958332047132 40.721488549007)
4,69482692,2026-06-26T02:03:29.000,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Crosswalk,NaN,Street/Sidewalk,11421.0,85 STREET,85 STREET,85 STREET,88 ROAD,85 STREET,88 ROAD,INTERSECTION,NaN,NaN,NaN,In Progress,NaN,NaN,NaN,09 QUEENS,32.0,Precinct 102,NaN,QUEENS,1023600.0,190652.0,ONLINE,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.689885,-73.858108,POINT (-73.858107579574 40.689884995274)


## Create Cleaned Copy

To protect the original raw dataset, I create a separate copy called `df_clean`.

All auto-fix functions will be applied to `df_clean`, not directly to `df`.

In [3]:
df_clean = df.copy()

print("Raw DataFrame shape:", df.shape)
print("Cleaned DataFrame shape:", df_clean.shape)

Raw DataFrame shape: (100000, 44)
Cleaned DataFrame shape: (100000, 44)


## Auto-Fix Function 1: Standardize Text

Many text columns contain inconsistent capitalization or extra spaces.

This function creates a reusable cleaning rule that removes leading/trailing spaces and converts text to uppercase while preserving missing values.

In [4]:
def standardize_text_column(series):
    """
    Removes leading/trailing spaces and converts text to uppercase.
    Missing values are preserved.
    """
    
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
    )

## Auto-Fix Function 2: Parse Date Columns

The raw dataset loads date-related columns as object/text.

This function converts selected date columns into datetime format so they can be used for time-based analysis.

In [5]:
def parse_date_columns(dataframe, date_columns):
    """
    Converts selected columns to datetime format.
    Invalid date values are converted to NaT.
    """
    
    dataframe = dataframe.copy()
    
    for column in date_columns:
        dataframe[column] = pd.to_datetime(dataframe[column], errors="coerce")
    
    return dataframe

## Auto-Fix Function 3: Clean ZIP Codes

The `incident_zip` column was loaded as a float, which creates values like `11211.0`.

ZIP codes are identifiers, not numeric measurements.

This function converts ZIP codes into clean string values while preserving missing values.

In [6]:
def clean_zip_code(value):
    """
    Converts ZIP code values from float format to clean string format.
    Missing values are preserved.
    """
    
    if pd.isna(value):
        return pd.NA
    
    return str(int(value))

## Auto-Fix Function 4: Remove Full-Row Duplicates

This function removes records that are completely duplicated across all columns.

Day 4 found no full-row duplicates in the current dataset, but this function makes DataGuard reusable for future raw data batches.

In [7]:
def remove_full_duplicates(dataframe):
    """
    Removes full-row duplicate records from a DataFrame.
    """
    
    return dataframe.drop_duplicates()

## Auto-Fix Function 6: Create Resolution Time

This function creates a `resolution_time` column using `closed_date - created_date`.

This field is useful for measuring how long service requests took to close.

Records with missing `closed_date` will have missing resolution time, which is expected for open or in-progress complaints.

In [8]:
def create_resolution_time(dataframe, start_column, end_column):
    """
    Creates a resolution_time column by subtracting the start date from the end date.
    """
    
    dataframe = dataframe.copy()
    dataframe["resolution_time"] = dataframe[end_column] - dataframe[start_column]
    
    return dataframe

## Apply Auto-Fix Functions

In this section, I apply the reusable cleaning functions to `df_clean`.

Each fix will also be documented in a list called `fixes_applied`, which will later become the Day 5 auto-fix report.

In [9]:
fixes_applied = []

print("Fix tracking list created.")

Fix tracking list created.


## Apply Fix 1: Standardize Text Columns

This fix standardizes selected text columns by removing extra spaces and converting values to uppercase.

In [10]:
text_columns = [
    "borough",
    "city",
    "status",
    "agency",
    "complaint_type",
    "descriptor"
]

for column in text_columns:
    df_clean[column] = standardize_text_column(df_clean[column])

fixes_applied.append({
    "fix_name": "Standardize text columns",
    "columns_affected": ", ".join(text_columns),
    "issue_fixed": "Inconsistent capitalization or extra spaces in text fields",
    "fix_applied": "str.strip() and str.upper()",
    "safe_auto_fix": "Yes",
    "notes": "Text values were standardized while preserving missing values."
})

df_clean[text_columns].head()

,borough,city,status,agency,complaint_type,descriptor
0,UNSPECIFIED,<NA>,IN PROGRESS,NYPD,NOISE - COMMERCIAL,LOUD MUSIC/PARTY
1,BROOKLYN,BROOKLYN,IN PROGRESS,TLC,FOR HIRE VEHICLE COMPLAINT,DRIVER COMPLAINT - NON PASSENGER
2,MANHATTAN,NEW YORK,IN PROGRESS,NYPD,NOISE - RESIDENTIAL,LOUD MUSIC/PARTY
3,BROOKLYN,BROOKLYN,IN PROGRESS,NYPD,NOISE - STREET/SIDEWALK,LOUD MUSIC/PARTY
4,QUEENS,<NA>,IN PROGRESS,NYPD,ILLEGAL PARKING,BLOCKED CROSSWALK


## Apply Fix 2: Parse Date Columns

This fix converts date-related columns from object/text format into datetime format.

Datetime format is needed for time-based analysis, including resolution-time calculations.

In [11]:
date_columns = [
    "created_date",
    "closed_date",
    "due_date",
    "resolution_action_updated_date"
]

print("Before date parsing:")
print(df_clean[date_columns].dtypes)

df_clean = parse_date_columns(df_clean, date_columns)

print("\nAfter date parsing:")
print(df_clean[date_columns].dtypes)

fixes_applied.append({
    "fix_name": "Parse date columns",
    "columns_affected": ", ".join(date_columns),
    "issue_fixed": "Date columns loaded as object/text",
    "fix_applied": "pd.to_datetime(errors='coerce')",
    "safe_auto_fix": "Yes",
    "notes": "Date columns were converted to datetime format for reliable time-based analysis."
})

Before date parsing:
created_date                      object
closed_date                       object
due_date                          object
resolution_action_updated_date    object
dtype: object

After date parsing:
created_date                      datetime64[ns]
closed_date                       datetime64[ns]
due_date                          datetime64[ns]
resolution_action_updated_date    datetime64[ns]
dtype: object


## Apply Fix 3: Clean ZIP Code Format

This fix creates a cleaned ZIP code column called `incident_zip_clean`.

The original `incident_zip` column was loaded as a float, which created values like `11211.0`.

ZIP codes are identifiers, so the cleaned version stores them as strings.

In [12]:
print("Before ZIP cleaning:")
print(df_clean["incident_zip"].head(10))

df_clean["incident_zip_clean"] = df_clean["incident_zip"].apply(clean_zip_code)

print("\nAfter ZIP cleaning:")
print(df_clean[["incident_zip", "incident_zip_clean"]].head(10))

fixes_applied.append({
    "fix_name": "Clean ZIP code format",
    "columns_affected": "incident_zip",
    "issue_fixed": "ZIP codes loaded as float values",
    "fix_applied": "Converted ZIP codes from float format to clean string format",
    "safe_auto_fix": "Yes",
    "notes": "Created incident_zip_clean while preserving the original incident_zip column."
})

Before ZIP cleaning:
0        NaN
1    11211.0
2    10013.0
3    11249.0
4    11421.0
5    11205.0
6    10456.0
7    11233.0
8    11205.0
9    11229.0
Name: incident_zip, dtype: float64

After ZIP cleaning:
   incident_zip incident_zip_clean
0           NaN               <NA>
1       11211.0              11211
2       10013.0              10013
3       11249.0              11249
4       11421.0              11421
5       11205.0              11205
6       10456.0              10456
7       11233.0              11233
8       11205.0              11205
9       11229.0              11229


## Apply Fix 4: Remove Full-Row Duplicates

This fix removes records that are completely duplicated across all columns.

Day 4 found no full-row duplicates, so the row count is expected to stay the same.

In [13]:
rows_before = df_clean.shape[0]

df_clean = remove_full_duplicates(df_clean)

rows_after = df_clean.shape[0]

duplicates_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Full-row duplicates removed:", duplicates_removed)

fixes_applied.append({
    "fix_name": "Remove full-row duplicates",
    "columns_affected": "All columns",
    "issue_fixed": "Fully duplicated records",
    "fix_applied": "drop_duplicates()",
    "safe_auto_fix": "Yes",
    "notes": f"Removed {duplicates_removed} full-row duplicate records."
})

Rows before: 100000
Rows after: 100000
Full-row duplicates removed: 0


## Apply Fix 5: Remove Duplicate Unique Keys

This fix removes duplicate records based on the `unique_key` column.

Each NYC 311 service request should have one unique key. Day 4 found no duplicate `unique_key` values, so the row count is expected to stay the same.

In [16]:
def remove_duplicate_keys(dataframe, key_column):
    """
    Removes duplicate records based on a key column.
    Keeps the first occurrence.
    """
    
    return dataframe.drop_duplicates(subset=[key_column], keep="first")

In [17]:
rows_before = df_clean.shape[0]

df_clean = remove_duplicate_keys(df_clean, "unique_key")

rows_after = df_clean.shape[0]

duplicate_keys_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Duplicate unique_key records removed:", duplicate_keys_removed)

fixes_applied.append({
    "fix_name": "Remove duplicate unique_key values",
    "columns_affected": "unique_key",
    "issue_fixed": "Duplicate service request identifiers",
    "fix_applied": "drop_duplicates(subset=['unique_key'], keep='first')",
    "safe_auto_fix": "Yes",
    "notes": f"Removed {duplicate_keys_removed} duplicate unique_key records."
})

Rows before: 100000
Rows after: 100000
Duplicate unique_key records removed: 0


## Apply Fix 6: Create Resolution Time

This fix creates a `resolution_time` column by subtracting `created_date` from `closed_date`.

This field will be used later for resolution-time analysis and service performance checks.

In [18]:
df_clean = create_resolution_time(df_clean, "created_date", "closed_date")

print("resolution_time column created.")
print(df_clean["resolution_time"].head(10))

fixes_applied.append({
    "fix_name": "Create resolution_time column",
    "columns_affected": "created_date, closed_date",
    "issue_fixed": "Resolution duration not directly available",
    "fix_applied": "closed_date - created_date",
    "safe_auto_fix": "Yes",
    "notes": "Created resolution_time for time-based analysis. Missing closed_date values result in missing resolution_time."
})

resolution_time column created.
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
5   NaT
6   NaT
7   NaT
8   NaT
9   NaT
Name: resolution_time, dtype: timedelta64[ns]


## Verify Resolution Time

The first few records may have missing `resolution_time` if `closed_date` is missing.

This check confirms how many records have valid resolution time and how many have negative resolution time.

In [19]:
valid_resolution_count = df_clean["resolution_time"].notnull().sum()
missing_resolution_count = df_clean["resolution_time"].isnull().sum()
negative_resolution_count = (df_clean["resolution_time"] < pd.Timedelta(0)).sum()

print("Valid resolution_time count:", valid_resolution_count)
print("Missing resolution_time count:", missing_resolution_count)
print("Negative resolution_time count:", negative_resolution_count)

Valid resolution_time count: 74905
Missing resolution_time count: 25095
Negative resolution_time count: 39


## Create Auto-Fix Report

This report documents each cleaning fix applied to the dataset.

The report shows what was fixed, which columns were affected, whether the fix was safe to automate, and notes about the cleaning step.

In [20]:
auto_fix_report = pd.DataFrame(fixes_applied)

print("Number of fixes documented:", len(auto_fix_report))
print("Report shape:", auto_fix_report.shape)

auto_fix_report

Number of fixes documented: 6
Report shape: (6, 6)


,fix_name,columns_affected,issue_fixed,fix_applied,safe_auto_fix,notes
0,Standardize text columns,"borough, city, status, agency, complaint_type,...",Inconsistent capitalization or extra spaces in...,str.strip() and str.upper(),Yes,Text values were standardized while preserving...
1,Parse date columns,"created_date, closed_date, due_date, resolutio...",Date columns loaded as object/text,pd.to_datetime(errors='coerce'),Yes,Date columns were converted to datetime format...
2,Clean ZIP code format,incident_zip,ZIP codes loaded as float values,Converted ZIP codes from float format to clean...,Yes,Created incident_zip_clean while preserving th...
3,Remove full-row duplicates,All columns,Fully duplicated records,drop_duplicates(),Yes,Removed 0 full-row duplicate records.
4,Remove duplicate unique_key values,unique_key,Duplicate service request identifiers,"drop_duplicates(subset=['unique_key'], keep='f...",Yes,Removed 0 duplicate unique_key records.
5,Create resolution_time column,"created_date, closed_date",Resolution duration not directly available,closed_date - created_date,Yes,Created resolution_time for time-based analysi...


In [21]:
data_folder = project_root / "data"
reports_folder = project_root / "reports"

data_folder.mkdir(exist_ok=True)
reports_folder.mkdir(exist_ok=True)

cleaned_data_path = data_folder / "nyc_311_cleaned_day5.csv"
auto_fix_report_path = reports_folder / "auto_fix_report_day5.csv"

df_clean.to_csv(cleaned_data_path, index=False)
auto_fix_report.to_csv(auto_fix_report_path, index=False)

print("Cleaned dataset saved to:", cleaned_data_path)
print("Auto-fix report saved to:", auto_fix_report_path)

Cleaned dataset saved to: c:\Users\samee\DataGuard\data\nyc_311_cleaned_day5.csv
Auto-fix report saved to: c:\Users\samee\DataGuard\reports\auto_fix_report_day5.csv


In [22]:
print("Cleaned data file exists:", cleaned_data_path.exists())
print("Auto-fix report file exists:", auto_fix_report_path.exists())

Cleaned data file exists: True
Auto-fix report file exists: True


## Final Validation Checks

Before completing Day 5, I run a few checks to confirm that the cleaned dataset and auto-fix report were created correctly.

In [24]:
print("Raw dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)

print("\nCleaned dataset columns added:")
new_columns = set(df_clean.columns) - set(df.columns)
print(new_columns)

print("\nAuto-fix report shape:", auto_fix_report.shape)

print("\nDate column dtypes:")
print(df_clean[date_columns].dtypes)

print("\nSample cleaned ZIP codes:")
print(df_clean[["incident_zip", "incident_zip_clean"]].head(10))

print("\nResolution time check:")
print("Valid resolution_time count:", df_clean["resolution_time"].notnull().sum())
print("Missing resolution_time count:", df_clean["resolution_time"].isnull().sum())
print("Negative resolution_time count:", (df_clean["resolution_time"] < pd.Timedelta(0)).sum())

Raw dataset shape: (100000, 44)
Cleaned dataset shape: (100000, 46)

Cleaned dataset columns added:
{'resolution_time', 'incident_zip_clean'}

Auto-fix report shape: (6, 6)

Date column dtypes:
created_date                      datetime64[ns]
closed_date                       datetime64[ns]
due_date                          datetime64[ns]
resolution_action_updated_date    datetime64[ns]
dtype: object

Sample cleaned ZIP codes:
   incident_zip incident_zip_clean
0           NaN               <NA>
1       11211.0              11211
2       10013.0              10013
3       11249.0              11249
4       11421.0              11421
5       11205.0              11205
6       10456.0              10456
7       11233.0              11233
8       11205.0              11205
9       11229.0              11229

Resolution time check:
Valid resolution_time count: 74905
Missing resolution_time count: 25095
Negative resolution_time count: 39


# Day 5 Summary

In this notebook, I created reusable auto-fix functions for simple, well-understood data quality issues in the NYC 311 dataset.

The raw dataset was preserved as `df`, and all cleaning was applied to a separate copied DataFrame called `df_clean`.

## Work Completed

- Loaded the original NYC 311 raw dataset
- Created a cleaned copy called `df_clean`
- Wrote a reusable function to standardize text columns
- Wrote a reusable function to parse date columns
- Wrote a reusable function to clean ZIP code formatting
- Wrote reusable functions to remove full-row duplicates and duplicate key values
- Wrote a reusable function to create `resolution_time`
- Applied the cleaning functions to `df_clean`
- Created a cleaned ZIP code column called `incident_zip_clean`
- Created a `resolution_time` column using `closed_date - created_date`
- Documented all applied fixes in an auto-fix report
- Exported the cleaned dataset
- Exported the Day 5 auto-fix report
- Ran final validation checks

## Key Results

- The raw dataset has 100,000 rows and 44 columns.
- The cleaned dataset has 100,000 rows and 46 columns.
- Two new columns were created: `incident_zip_clean` and `resolution_time`.
- Date columns were converted from object/text format to datetime format.
- ZIP codes were converted from float format into clean string values.
- 0 full-row duplicate records were removed.
- 0 duplicate `unique_key` records were removed.
- 74,905 records have valid `resolution_time`.
- 25,095 records have missing `resolution_time`, mostly because `closed_date` is missing for open or in-progress requests.
- 39 records still have negative `resolution_time`, which should be flagged for review instead of silently fixed.

## Outputs Created

The cleaned dataset was saved to:

`data/nyc_311_cleaned_day5.csv`

The auto-fix report was saved to:

`reports/auto_fix_report_day5.csv`

## Next Step

The next step is to validate the cleaned dataset and begin turning notebook logic into reusable project scripts.